In [23]:
import json
import re
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from nltk.stem import WordNetLemmatizer

In [2]:
def data_load(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            appid, info = next(iter(item.items()))
            info['appid'] = appid
            data.append(info)
    return data

In [3]:
def df_cleansing(data_to_df):
    df = pd.DataFrame(data_to_df)
    
    # 0. Filter non-English games early using 'supported_languages'
    # This filters out purely foreign (e.g., Chinese) games before expensive processing
    if 'supported_languages' in df.columns:
        df = df[df['supported_languages'].str.contains('English', case=False, na=False)]
    
    # 1. Drop columns in one vectorized operation without loops
    cols_to_drop = ['header_image', 'reviews', 'supported_languages', 'support_info', 'game_link', 'appid', 'demos', 'ext_user_account_notice', 'drm_notice']
    df = df.drop(columns=cols_to_drop, errors='ignore')

    # 2. Use pure pandas .str.replace chain
    pattern = r'<[^>]*>'
    df['about_the_game'] = df['about_the_game'].str.replace(pattern, ' ', regex=True).str.replace(r'\n', ' ', regex=True)

    # 3. Clean to lists.
    def clean_list(x):
        return [str(i).strip() for i in x if str(i).strip()] if isinstance(x, list) else []

    def clean_dict_list(x):
        return [d.get('description', '').strip() for d in x if d.get('description', '').strip()] if isinstance(x, list) else []

    df['developers'] = df['developers'].apply(clean_list)
    df['publishers'] = df['publishers'].apply(clean_list)
    df['genres'] = df['genres'].apply(clean_dict_list)
    df['categories'] = df['categories'].apply(clean_dict_list)

    df = df.reset_index(drop=True)

    # 4. Vectorized boolean mapping
    df['is_free'] = (df['is_free'] == True).astype(int)

    # 5. Extract MultiLabelBinarizer results in memory and concat once
    def get_mlb_df(column_name):
        mlb = MultiLabelBinarizer()
        encoded_matrix = mlb.fit_transform(df[column_name])
        return pd.DataFrame(encoded_matrix, columns=mlb.classes_, index=df.index)

    mlb_dfs = [
        get_mlb_df('genres'),
        get_mlb_df('categories'),
        get_mlb_df('developers'),
        get_mlb_df('publishers')
    ]
    
    # Drop original categorical columns
    df = df.drop(columns=['genres', 'categories', 'developers', 'publishers'])
    
    df = pd.concat([df] + mlb_dfs, axis=1)
    
    df = df.drop(columns=['Free To Play'], errors='ignore')

    df['dlc_count'] = df['dlc'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    df['achievements_count'] = df['achievements'].apply(lambda x: x.get('total', 0) if isinstance(x, dict) else 0)
    df = df.drop(columns=['dlc', 'achievements'], errors='ignore')

    # 6. Vectorized datetime parsing (dramatically faster than .apply try/excepts)
    release_str = df['release_date'].apply(lambda x: x.get('date', '') if isinstance(x, dict) else '')
    df['release_year'] = pd.to_datetime(release_str, errors='coerce').dt.year
    df = df.drop(columns=['release_date'], errors='ignore')

    if 'controller_support' in df.columns:
        df['controller_support'] = df['controller_support'].fillna('0').astype(str).str.replace('full', '1')

    if 'recommendations' in df.columns:
        df['recommendations'] = df['recommendations'].apply(lambda x: x.get('total', 0) if isinstance(x, dict) else 0)
    
    df['price_overview'] = df['price_overview'].apply(lambda x: float(x.get('initial', 0)) if isinstance(x, dict) else 0.0)
    
    return df

In [4]:
data_example = data_load(file_path='../data/games_informations/example.jsonl')
dataframe = df_cleansing(data_to_df=data_example)
dataframe = dataframe.set_index('name')
# dataframe.to_excel('ex.xlsx')

## nlp 

In [5]:
about = dataframe['about_the_game']
print(about.head(2), type(about))

name
Mystic Space    Mystic Space is a fast-paced high action game ...
DK Online         DK Online - Season 4: Annihilation   Opening...
Name: about_the_game, dtype: str <class 'pandas.Series'>


In [8]:
# Re-extract 'about' to guarantee it exactly matches the dataframe's current length
about = dataframe['about_the_game'].astype(str)

lemmatizer = WordNetLemmatizer()
data_clean = []
for cell in about:
    cell = cell.lower()
    tokens = re.findall(r'\b[a-z0-9\-]+\b', cell)
    cell_clean = ' '.join(lemmatizer.lemmatize(word) for word in tokens)
    data_clean.append(cell_clean)

tfidf = TfidfVectorizer(stop_words='english', max_features=1000, min_df=10, max_df=0.85)
data_tfidf = tfidf.fit_transform(data_clean)

# Convert sparse matrix to dense array and assign feature names to columns
df_tfidf = pd.DataFrame(
    data_tfidf.toarray(), 
    columns=tfidf.get_feature_names_out(),
    index=dataframe.index  # Keep the original index
)

# Use concat instead of merge! pd.merge on an index with duplicates creates a cartesian product (size explosion)
dataframe_final = pd.concat([dataframe.drop('about_the_game', axis=1, errors='ignore'), df_tfidf], axis=1)

dataframe_final.to_excel('../data/analysis_example_data/final.xlsx')

              is_free  price_overview  recommendations controller_support  \
name                                                                        
Mystic Space        0           899.0                0                  0   
DK Online           1             0.0                0                  0   

              Action  Adventure  Casual  Early Access  Gore  Indie  ...  word  \
name                                                                ...         
Mystic Space       1          1       1             0     0      1  ...   0.0   
DK Online          1          1       0             0     0      0  ...   0.0   

              work  working  workshop     world  worry  year  young  zombie  \
name                                                                          
Mystic Space   0.0      0.0       0.0  0.000000    0.0   0.0    0.0     0.0   
DK Online      0.0      0.0       0.0  0.122461    0.0   0.0    0.0     0.0   

              zone  
name                
Mystic 

# scaling

In [28]:
cols_to_scale = ['price_overview', 'recommendations', 'dlc_count', 'achievements_count']

scaler = MinMaxScaler()

def scale(dataframe, column):
    dataframe[column] = scaler.fit_transform(dataframe[[column]])

for col in cols_to_scale:
    scale(dataframe_final, col)

dataframe_final.to_excel('../data/analysis_example_data/final_scaled.xlsx')



In [27]:
print(dataframe_final.head(2), dataframe_final.shape)

              is_free  price_overview  recommendations controller_support  \
name                                                                        
Mystic Space        0        0.105044              0.0                  0   
DK Online           1        0.000000              0.0                  0   

              Action  Adventure  Casual  Early Access  Gore  Indie  ...  word  \
name                                                                ...         
Mystic Space       1          1       1             0     0      1  ...   0.0   
DK Online          1          1       0             0     0      0  ...   0.0   

              work  working  workshop     world  worry  year  young  zombie  \
name                                                                          
Mystic Space   0.0      0.0       0.0  0.000000    0.0   0.0    0.0     0.0   
DK Online      0.0      0.0       0.0  0.122461    0.0   0.0    0.0     0.0   

              zone  
name                
Mystic 